# RLHF on Kaggle — run once and walk away

**Fire-and-forget.** No monitoring, no per-cell editing.

1. **Settings**: Accelerator = **GPU T4 ×2** (or P100), Internet = **On**.
2. (Optional) set `MODEL` / `PRESET` / `DATASET` in the **config** cell.
3. **Save Version ▸ Save & Run All (Commit)** → close the tab.  *(Or headless from a terminal:
   `scripts/run_on_kaggle.sh` via the Kaggle API.)*

Runs SFT → reward model → PPO → eval and writes **`RESULTS.md`** (RM accuracy, PPO-vs-SFT win-rate,
samples) to the output.

> Free GPUs can be **preempted** (the run restarts from scratch). A `full` 0.5B run is ~3–5 h; if it
> keeps restarting use `PRESET='fast'` or split the stages across separate commits.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable the accelerator!')
!pip install -q -U 'transformers>=4.44' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12' pytest

In [ ]:
import os
REPO_URL = 'https://github.com/TheYellowDuck/RLHF-pipeline.git'
ATTACHED = '/kaggle/input/rlhf-pipeline'
if REPO_URL:
    !rm -rf /kaggle/working/rlhf-pipeline && git clone -q $REPO_URL /kaggle/working/rlhf-pipeline
elif os.path.exists(ATTACHED):
    !rm -rf /kaggle/working/rlhf-pipeline && cp -r $ATTACHED /kaggle/working/rlhf-pipeline
else:
    raise SystemExit('Set REPO_URL or attach the repo as a Kaggle Dataset.')
%cd /kaggle/working/rlhf-pipeline

In [ ]:
# Pre-flight: proves the pipeline is sound on this box (tiny model, ~1 min). Aborts the run if red.
!python scripts/smoke_test.py && python -m pytest tests/ -q

## Config — the only knobs

In [ ]:
MODEL   = 'Qwen/Qwen2.5-0.5B'   # coherent text; smaller/faster: 'EleutherAI/pythia-160m'
PRESET  = 'full'                # 'fast' (~2 h)  or  'full' (~3-5 h on 0.5B)
DATASET = 'uf'                  # 'hh' = Anthropic/hh-rlhf (classic RLHF) | 'uf' = UltraFeedback (cleaner -> higher RM accuracy)

if DATASET == 'uf':
    DATA, TRAIN_SPLIT, EVAL_SPLIT = 'HuggingFaceH4/ultrafeedback_binarized', 'train_prefs', 'test_prefs'
else:
    DATA, TRAIN_SPLIT, EVAL_SPLIT = 'Anthropic/hh-rlhf', 'train', 'test'

if PRESET == 'fast':
    SFT_SAMPLES, RM_SAMPLES, PPO_EPISODES, MAX_NEW = 4000, 8000, 1024, 32
else:
    SFT_SAMPLES, RM_SAMPLES, PPO_EPISODES, MAX_NEW = 8000, 20000, 2048, 40
ROLLOUT, MINI, PPO_LR = 16, 4, '5e-7'
print(f'model={MODEL}  preset={PRESET}  data={DATA}  sft={SFT_SAMPLES}  rm={RM_SAMPLES}  ppo_ep={PPO_EPISODES}')

## 1. SFT

In [ ]:
!python scripts/train_sft.py \
  -o model.name_or_path=$MODEL -o data.name=$DATA -o data.train_split=$TRAIN_SPLIT -o data.eval_split=$EVAL_SPLIT \
  -o data.max_samples=$SFT_SAMPLES -o train.batch_size=8 -o train.bf16=true -o output_dir=checkpoints/sft

## 2. Reward model  (init from SFT + label-smoothed BT loss + larger effective batch)

In [ ]:
!python scripts/train_reward_model.py \
  -o model.name_or_path=checkpoints/sft -o data.name=$DATA -o data.train_split=$TRAIN_SPLIT -o data.eval_split=$EVAL_SPLIT \
  -o data.max_samples=$RM_SAMPLES -o data.max_eval_samples=500 \
  -o train.batch_size=8 -o train.bf16=true -o output_dir=checkpoints/reward_model

## 3. PPO (RL against the reward model)

In [ ]:
!python scripts/train_ppo.py \
  -o policy.name_or_path=checkpoints/sft -o reward_model.name_or_path=checkpoints/reward_model \
  -o data.name=$DATA -o data.train_split=$TRAIN_SPLIT \
  -o ppo.total_episodes=$PPO_EPISODES -o ppo.rollout_batch_size=$ROLLOUT -o ppo.mini_batch_size=$MINI \
  -o ppo.lr=$PPO_LR -o ppo.normalize_rewards=true \
  -o generation.max_new_tokens=$MAX_NEW -o data.max_prompt_length=160

## 4. Results → written to RESULTS.md (no API key needed)

In [ ]:
import subprocess
def run(cmd):
    print('$', cmd, flush=True)
    o = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(o.stdout)
    if o.returncode: print(o.stderr[-1500:])
    return o.stdout

rm  = run(f'python scripts/evaluate.py rm-accuracy --reward-model checkpoints/reward_model --data {DATA} --split {EVAL_SPLIT} --max-samples 500')
win = run(f'python scripts/evaluate.py score-policy --policy checkpoints/ppo --reward-model checkpoints/reward_model --compare checkpoints/sft --data {DATA} --split {TRAIN_SPLIT} --num 100 --max-new-tokens {MAX_NEW}')

md = (f'# RLHF run results\n\n- model: `{MODEL}`\n- dataset: `{DATA}`\n- preset: `{PRESET}`\n\n'
      f'## Reward-model accuracy (held-out)\n```\n{rm}\n```\n\n'
      f'## PPO vs SFT — reward-model win-rate + sample completions\n```\n{win}\n```\n')
open('/kaggle/working/RESULTS.md','w').write(md)
print('\nSaved -> /kaggle/working/RESULTS.md')

### When it finishes
Output tab → **`RESULTS.md`** + `rlhf-pipeline/checkpoints/` (models + `metrics.jsonl` curves).

**Raising RM accuracy** (current levers already on: RM-from-SFT init, label-smoothed loss, larger batch):
- `DATASET='uf'` — UltraFeedback is cleaner/higher-contrast than HH-RLHF → the most reliable accuracy bump.
- `-o train.epochs=2` on the RM cell — multi-epoch RMs can help (~2× RM time).
- Realistic ceiling on HH-RLHF is ~0.66–0.70 (label noise); UltraFeedback tends higher.

**OOM?** add `-o policy.use_lora=true` / `-o model.use_lora=true`, or lower `ROLLOUT`/`MINI`.